## MLflow's Model Registry

In [1]:
from mlflow.tracking import MlflowClient


MLFLOW_TRACKING_URI = "sqlite:///mlflow.db"

### Interacting with the MLflow tracking server

The `MlflowClient` object allows us to interact with...
- an MLflow Tracking Server that creates and manages experiments and runs.
- an MLflow Registry Server that creates and manages registered models and model versions. 

To instantiate it we need to pass a tracking URI and/or a registry URI


**MLflow 2.0**

La vidéo du cours a été enregistrée avec MLflow 1.x, où la méthode s'appelait `list_experiments()`. Depuis MLflow 2.0, elle a été supprimée et remplacée par `search_experiments()`.

La logique du changement : MLflow a unifié toutes les méthodes de consultation sous le préfixe `search_`. Tu retrouves la même famille avec `search_runs()`, `search_model_versions()`, etc. Une méthode `search_` peut faire ce que faisait `list_` (tout renvoyer) *et* accepter des filtres en plus.

**La correction**

```python
client.search_experiments()
```

Sans argument, le comportement est identique à l'ancien `list_experiments()` : elle te renvoie tous les expériments actifs.

**Le bonus du changement**

Puisque c'est une recherche, tu peux filtrer directement, ce qui n'était pas possible avant :

```python
# filtrer par nom
client.search_experiments(filter_string="name LIKE 'nyc%'")

# inclure aussi les expériments supprimés
client.search_experiments(view_type=ViewType.ALL)
```

`ViewType` s'importe depuis `mlflow.entities` :

```python
from mlflow.entities import ViewType
```

Tu vas en avoir besoin juste après dans le notebook du cours, pour la partie `search_runs()`.

**Le réflexe à garder**

Tu vas rencontrer plusieurs fois ce type de décalage entre la vidéo et ta version installée. Le réflexe utile : quand une méthode "n'existe pas", vérifie d'abord si elle a été renommée avant de chercher un problème ailleurs. Tu peux lister ce qui existe vraiment sur ton objet :

```python
[m for m in dir(client) if 'experiment' in m]
```

Ça t'aurait montré `search_experiments` tout de suite. C'est une petite technique qui dépanne souvent.


**Point 1 : `dir()` ne renvoie pas que des méthodes**

`dir(client)` renvoie une liste de **chaînes de caractères** — les noms de tout ce qui est accessible sur l'objet. Ça inclut les méthodes, mais aussi les attributs (des données simples) et les `__trucs__` internes de Python.

Deux conséquences pratiques :

- Tu récupères des noms, pas les fonctions elles-mêmes. `'search_experiments'` est un texte. Pour appeler la méthode, il faut toujours écrire `client.search_experiments()`.
- Le filtre `if 'experiment' in m` teste donc une sous-chaîne dans un texte, comme un `Ctrl+F`.

**Point 2 : la casse compte**

`'experiment' in m` est sensible à la casse. Un nom comme `getExperiment` ou `ExperimentInfo` ne serait pas trouvé. Version plus robuste :

```python
[m for m in dir(client) if 'experiment' in m.lower()]
```

**Décomposition de la syntaxe**

La compréhension de liste se lit de l'intérieur vers l'extérieur :

```python
[m           for m in dir(client)        if 'experiment' in m]
#  ↑              ↑                              ↑
# ce que      sur quoi                      condition
# je garde    je boucle                     de filtrage
```

L'équivalent en boucle classique, strictement identique :

```python
resultat = []
for m in dir(client):
    if 'experiment' in m:
        resultat.append(m)
```

**Pourquoi c'est utile ici**

C'est ton outil d'exploration quand une méthode "a disparu". Tu ne cherches pas dans la doc en ligne (qui peut correspondre à une autre version) — tu interroges **l'objet réel installé sur ta machine**. C'est la seule source de vérité sur ta version.

Deux variantes que tu utiliseras souvent :

```python
# tout sauf les méthodes internes de Python
[m for m in dir(client) if not m.startswith('_')]

# lire la documentation d'une méthode précise
help(client.search_experiments)
```

Le `help()` t'affiche la signature et les paramètres attendus, directement depuis ta version installée. Très pratique quand la vidéo du cours ne correspond plus.

In [5]:
client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

#client.list_experiments()
client.search_experiments()

[<Experiment: artifact_location='/workspaces/mlops-zoomcamp/02-experiment-tracking/mlruns/1', creation_time=1788898117863, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1788898117863, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='mlflow-artifacts:/0', creation_time=1788895016078, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1788895016078, lifecycle_stage='active', name='Default', tags={}, trace_location=None, workspace='default'>]

In [6]:
[m for m in dir(client) if 'experiment' in m]

['_link_prompt_to_experiment',
 'add_dataset_to_experiments',
 'create_experiment',
 'delete_experiment',
 'delete_experiment_tag',
 'get_experiment',
 'get_experiment_by_name',
 'remove_dataset_from_experiments',
 'rename_experiment',
 'restore_experiment',
 'search_experiments',
 'set_experiment_tag']

In [7]:
client.create_experiment(name="my-cool-experiment")

'2'

**`ViewType` est une énumération** qui sert à filtrer les runs selon leur **état de suppression**.

Elle a trois valeurs possibles :

| Valeur | Ce que ça retourne |
|---|---|
| `ViewType.ACTIVE_ONLY` | Uniquement les runs actifs (comportement par défaut) |
| `ViewType.DELETED_ONLY` | Uniquement les runs mis à la corbeille |
| `ViewType.ALL` | Les deux |

---

**Pourquoi ça existe ?**

Parce que dans MLflow, supprimer un run ne l'efface pas vraiment. C'est un **soft delete** : le run est marqué `deleted` en base, mais ses données restent là.

Concrètement, quand tu cliques sur la poubelle dans l'UI MLflow, le run disparaît de la liste mais il est toujours dans ton `mlflow.db`. C'est la même logique que la corbeille de ton système d'exploitation.

Ça permet de restaurer un run supprimé par erreur :

```python
client.restore_run(run_id)
```

---

**Dans ton code**

```python
run_view_type=ViewType.ACTIVE_ONLY
```

Tu demandes explicitement à ne voir que les runs vivants. C'est déjà le comportement par défaut, donc la ligne est surtout là pour être explicite et pédagogique.

Si tu voulais retrouver ce que tu as supprimé :

```python
runs = client.search_runs(
    experiment_ids='1',
    run_view_type=ViewType.DELETED_ONLY
)
```

---

**Le reste des arguments, pour être complet**

- `experiment_ids='1'` → dans quel experiment chercher (l'ID, pas le nom)
- `filter_string="metrics.rmse < 7"` → un filtre en syntaxe SQL-like ; tu peux aussi filtrer sur `params.*`, `tags.*`, `attributes.*`
- `max_results=5` → nombre de runs retournés
- `order_by=["metrics.rmse ASC"]` → tri croissant, donc les 5 **meilleurs** runs

Combiné, ça donne : *"les 5 meilleurs runs actifs de l'experiment 1 ayant un RMSE inférieur à 7"*.

Let's check the latest versions for the experiment with id `1`...

In [8]:
from mlflow.entities import ViewType

runs = client.search_runs(
    experiment_ids='1',
    filter_string="metrics.rmse < 7",
    run_view_type=ViewType.ACTIVE_ONLY,
    max_results=5,
    order_by=["metrics.rmse ASC"]
)

In [9]:
for run in runs:
    print(f"run id: {run.info.run_id}, rmse: {run.data.metrics['rmse']:.4f}")

run id: 7e950e1b717f4a25977d9cd555fbc077, rmse: 6.3184
run id: c64c1d9edc434ba89ff48f04eec9803b, rmse: 6.4359
run id: 5dd42d706e514ceb94e40cb0f3f31dc0, rmse: 6.5549


### Interacting with the Model Registry

In this section We will use the `MlflowClient` instance to:

1. Register a new version for the experiment `nyc-taxi-regressor`
2. Retrieve the latests versions of the model `nyc-taxi-regressor` and check that a new version `4` was created.
3. Transition the version `4` to "Staging" and adding annotations to it.

In [10]:
import mlflow

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

MLflow te dit : dans ce run, il n'y a rien d'enregistré comme *modèle* au chemin `model`. Le run existe, mais le contenu attendu n'y est pas.

**La distinction à avoir en tête**

Il y a deux façons très différentes de mettre un fichier dans un run :

| | Commande | Résultat |
|---|---|---|
| Artefact simple | `mlflow.log_artifact(...)` | un fichier joint au run, MLflow ne sait pas ce que c'est |
| Modèle | `mlflow.sklearn.log_model(...)`, `mlflow.xgboost.log_model(...)` | un modèle structuré, avec sa signature et son environnement |

Seul le second est enregistrable dans le Model Registry. `register_model` refuse le premier.

Or dans ta cellule Lasso de tout à l'heure, tu faisais uniquement :

```python
mlflow.log_artifact(local_path="models/lin_reg.bin", artifact_path="models_pickle")
```

C'est un pickle brut posé à côté du run. Pour MLflow, c'est un fichier quelconque, pas un modèle.

**Diagnostic : regarde ce que contient vraiment ce run**

```python
client.list_artifacts(run_id)
```

Trois cas possibles :

1. Tu vois `models_mlflow` → c'est juste le nom du chemin qui diffère. Corrige l'URI :
   ```python
   model_uri = f"runs:/{run_id}/models_mlflow"
   ```
   C'est le nom utilisé dans le notebook du cours pour le run XGBoost.

2. Tu vois seulement `models_pickle` → ce run n'a pas de modèle enregistrable. Il faut en choisir un autre, ou relancer un entraînement avec `log_model`.

3. Tu ne vois rien de pertinent → mauvais `run_id`.

**Vérification complémentaire (MLflow 3)**

Ta version utilise la notion de *logged model*, qui est plus récente. Tu peux lister directement ce qui est enregistré comme modèle :

```python
mlflow.search_logged_models(filter_string=f"source_run_id = '{run_id}'")
```

Si ça renvoie un tableau vide, c'est confirmé : ce run ne contient aucun modèle au sens MLflow.

**Pour produire un run enregistrable**

Reprends ta cellule Lasso en remplaçant le `log_artifact` par un vrai `log_model` :

```python
with mlflow.start_run():
    mlflow.set_tag("developer", "pierre")
    alpha = 0.1
    mlflow.log_param("alpha", alpha)

    lr = Lasso(alpha)
    lr.fit(X_train, y_train)

    y_pred = lr.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mlflow.log_metric("rmse", rmse)

    mlflow.sklearn.log_model(lr, name="model")
```

Un détail de version : dans MLflow 3, le paramètre s'appelle désormais `name=` et non plus `artifact_path=`. L'ancien nom fonctionne encore mais affiche un avertissement de dépréciation.

Récupère ensuite le nouveau `run_id` affiché dans l'UI, et relance ton `register_model` avec.

Mon diagnostic était faux sur un point important : ce run **contient bien** un vrai modèle MLflow. Je m'attendais à un tableau vide, ce n'est pas le cas. Voilà ce que ton résultat dit réellement.

**Lecture de la ligne**

```
1    c64c1d9edc434ba89ff48f04eec9803b    models:/m-6bd8536277004e9683ef5a6297468514
│    │                                    │
version  run_id (le tien !)               source
```

Deux enseignements :

1. Le `run_id` est **exactement celui de ton code**. Tu ne t'étais pas trompé de run.
2. Le `source` n'est pas du tout de la forme `runs:/<run_id>/model`. C'est `models:/m-...`.

**Ce que ça révèle : MLflow 3 a changé de modèle interne**

En MLflow 2, un modèle était un simple dossier d'artefacts rangé dans un run. On y accédait par son chemin : `runs:/<run_id>/<artifact_path>`.

En MLflow 3 (tu es en 3.16.0), le modèle est devenu une **entité à part entière**, avec son propre identifiant `m-...`, indépendant du run. Le run n'est plus qu'un lien de parenté (`source_run_id`).

Quand tu écris `runs:/<run_id>/model`, MLflow 3 fait une traduction de compatibilité : il cherche, parmi les logged models de ce run, celui dont le nom est `model`. Il n'en trouve pas — donc le nom sous lequel ton modèle a été enregistré est autre chose.

**La solution**

Utilise directement l'identifiant, sans passer par la traduction :

```python
mlflow.register_model(
    model_uri="models:/m-6bd8536277004e9683ef5a6297468514",
    name="nyc-taxi-regressor"
)
```

Ça marchera et créera une Version 2.

**Pour retrouver le nom réel**

Si tu veux comprendre pourquoi `model` ne correspondait pas :

```python
for m in mlflow.search_logged_models(
    filter_string=f"source_run_id = '{run_id}'",
    output_format="list"
):
    print(m.name, "|", m.model_id)
```

Le `m.name` affiché est celui à mettre dans `runs:/{run_id}/{name}` si tu tiens à garder cette syntaxe. C'est probablement `models_mlflow`, le nom utilisé dans le notebook du cours.

**Ce qu'il faut retenir pour la suite**

Tu n'as rien à refaire : ta Version 1 existe et est correcte. La registration manuelle a fonctionné parce que l'UI utilise directement le `model_id`, sans passer par la traduction de chemin.

Le réflexe à garder : en MLflow 3, quand une URI `runs:/...` échoue, va chercher le `model_id` et utilise `models:/m-...`. C'est la voie directe, celle qui ne dépend pas d'un nom de chemin.

In [20]:
for v in client.search_model_versions("name='nyc-taxi-regressor'"):
    print(v.version, v.run_id, v.source)

1 c64c1d9edc434ba89ff48f04eec9803b models:/m-6bd8536277004e9683ef5a6297468514


In [23]:
run_id = "c64c1d9edc434ba89ff48f04eec9803b"
model_uri = f"runs:/{run_id}/model"

client.list_artifacts(run_id)
print(client.list_artifacts(run_id))
#mlflow.register_model(model_uri=model_uri, name="nyc-taxi-regressor")
mlflow.register_model(
    model_uri="models:/m-6bd8536277004e9683ef5a6297468514",
    name="nyc-taxi-regressor"
)

[<FileInfo: file_size=None, is_dir=True, path='preprocessor'>]


Registered model 'nyc-taxi-regressor' already exists. Creating a new version of this model...
Created version '2' of model 'nyc-taxi-regressor'.


<ModelVersion: aliases=[], creation_timestamp=1789415745541, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1789415745541, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='c64c1d9edc434ba89ff48f04eec9803b', run_link=None, source='models:/m-6bd8536277004e9683ef5a6297468514', status='READY', status_message=None, tags={}, user_id=None, version=2, workspace='default'>

In [22]:
print(client.list_artifacts(run_id))

[<FileInfo: file_size=None, is_dir=True, path='preprocessor'>]


In [25]:
model_name = "nyc-taxi-regressor"
latest_versions = client.get_latest_versions(name=model_name)

for version in latest_versions:
    print(f"version: {version.version}, stage: {version.current_stage}")

version: 2, stage: None


/tmp/ipykernel_6444/669935608.py:2: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  latest_versions = client.get_latest_versions(name=model_name)


In [32]:
model_version = 4
new_stage = "Staging"
client.transition_model_version_stage(
    name=model_name,
    version=model_version,
    stage=new_stage,
    archive_existing_versions=False
)

<ModelVersion: creation_timestamp=1652971637398, current_stage='Staging', description='The model version 4 was transitioned to Staging on 2022-05-19', last_updated_timestamp=1652972141519, name='nyc-taxi-regressor', run_id='b8904012c84343b5bf8ee72aa8f0f402', run_link=None, source='./mlruns/1/b8904012c84343b5bf8ee72aa8f0f402/artifacts/model', status='READY', status_message=None, tags={}, user_id=None, version=4>

In [33]:
from datetime import datetime

date = datetime.today().date()
client.update_model_version(
    name=model_name,
    version=model_version,
    description=f"The model version {model_version} was transitioned to {new_stage} on {date}"
)

<ModelVersion: creation_timestamp=1652971637398, current_stage='Staging', description='The model version 4 was transitioned to Staging on 2022-05-19', last_updated_timestamp=1652972142779, name='nyc-taxi-regressor', run_id='b8904012c84343b5bf8ee72aa8f0f402', run_link=None, source='./mlruns/1/b8904012c84343b5bf8ee72aa8f0f402/artifacts/model', status='READY', status_message=None, tags={}, user_id=None, version=4>

### Comparing versions and selecting the new "Production" model

In the last section, we will retrieve models registered in the model registry and compare their performance on an unseen test set. The idea is to simulate the scenario in which a deployment engineer has to interact with the model registry to decide whether to update the model version that is in production or not.

These are the steps:

1. Load the test dataset, which corresponds to the NYC Green Taxi data from the month of March 2021.
2. Download the `DictVectorizer` that was fitted using the training data and saved to MLflow as an artifact, and load it with pickle.
3. Preprocess the test set using the `DictVectorizer` so we can properly feed the regressors.
4. Make predictions on the test set using the model versions that are currently in the "Staging" and "Production" stages, and compare their performance.
5. Based on the results, update the "Production" model version accordingly.


**Note: the model registry doesn't actually deploy the model to production when you transition a model to the "Production" stage, it just assign a label to that model version. You should complement the registry with some CI/CD code that does the actual deployment.**

In [34]:
from sklearn.metrics import mean_squared_error
import pandas as pd


def read_dataframe(filename):
    df = pd.read_csv(filename)

    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    return df


def preprocess(df, dv):
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    categorical = ['PU_DO']
    numerical = ['trip_distance']
    train_dicts = df[categorical + numerical].to_dict(orient='records')
    return dv.transform(train_dicts)


def test_model(name, stage, X_test, y_test):
    model = mlflow.pyfunc.load_model(f"models:/{name}/{stage}")
    y_pred = model.predict(X_test)
    return {"rmse": mean_squared_error(y_test, y_pred, squared=False)}

In [36]:
df = read_dataframe("data/green_tripdata_2021-03.csv")

/var/folders/42/f9s_rgk15078ym2w50_xtc180000gq/T/ipykernel_5486/3050441246.py:6: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filename)


In [37]:
client.download_artifacts(run_id=run_id, path='preprocessor', dst_path='.')

'/Users/cristian.martinez/Repositories/mlops-zoomcamp/02-experiment-tracking/preprocessor'

**Tu es tout près, mais il y a une confusion importante à lever.**

---

### Ce qu'on charge n'est PAS de la data

`preprocessor.b` ne contient **aucune donnée de trajet**. Il contient l'objet `dv` lui-même, c'est-à-dire **l'outil** qui sait transformer des données.

Souviens-toi de comment il a été créé :

```python
with open("models/preprocessor.b", "wb") as f_out:
    pickle.dump(dv, f_out)     # ← on sauvegarde dv, pas X_train !
```

On a sérialisé le `DictVectorizer`, pas les matrices.

---

### Distinction fondamentale

| | Contenu |
|---|---|
| `X_train`, `X_val` | Les **données** transformées (matrices de chiffres) |
| `dv` | La **recette de transformation** (le vocabulaire appris) |

Le fichier contient la deuxième chose.

---

### Que contient exactement `dv` ?

Le résultat du `fit` dont on a longuement parlé — la correspondance entre les valeurs catégorielles et les positions de colonnes :

```python
dv.vocabulary_
# {'PU_DO=130_85': 0,
#  'PU_DO=41_74': 1,
#  'PU_DO=22_53': 2,
#  ...
#  'trip_distance': 5341}
```

C'est ça qu'on sauvegarde : *"la zone 130_85 va en colonne 0, la zone 41_74 en colonne 1..."*

---

### À quoi ça sert concrètement ?

C'est **le cœur du problème du déploiement**. Imagine que ton modèle tourne en production dans six mois, dans un autre programme, sur une autre machine. Un nouveau trajet arrive :

```python
nouveau_trajet = [{'PU_DO': '41_74', 'trip_distance': 3.2}]
```

Tu ne peux pas le donner tel quel au modèle — il attend une matrice de plusieurs milliers de colonnes, dans un ordre bien précis. Il te faut **exactement le même** DictVectorizer que celui utilisé à l'entraînement :

```python
X_nouveau = dv.transform(nouveau_trajet)   # transform, jamais fit !
y_pred = booster.predict(xgb.DMatrix(X_nouveau))
```

Si tu créais un `DictVectorizer()` neuf, il n'aurait aucun vocabulaire → les colonnes ne correspondraient plus → prédictions absurdes. C'est exactement le problème dont on parlait avec le formulaire d'impôts.

---

### Petite précision sur ta formulation

> on charge le fichier preprocessor.b dans le fichier f_in

Pas tout à fait : `f_in` n'est pas un fichier, c'est une **poignée** vers le fichier ouvert — un objet Python qui permet d'y lire. Le contenu, lui, atterrit dans `dv` grâce à `pickle.load(f_in)`.

```python
open(...)      → ouvre le robinet, f_in est la poignée
pickle.load()  → aspire le contenu et le reconstruit en objet Python
```

---

### Le schéma complet du déploiement

```
preprocessor.b  →  dv       ─┐
                             ├→  données brutes → prédiction
modèle MLflow   →  booster  ─┘
```

Les deux sont indispensables, et c'est pour ça que le cours insiste pour logger le preprocessor comme artifact à côté du modèle.


In [38]:
import pickle

with open("preprocessor/preprocessor.b", "rb") as f_in:
    dv = pickle.load(f_in)

In [39]:
X_test = preprocess(df, dv)

In [40]:
target = "duration"
y_test = df[target].values

**`%` indique une "magic command" de Jupyter/IPython.**

Ce n'est pas du Python standard. Si tu copiais cette ligne dans un fichier `.py` classique, elle planterait avec une erreur de syntaxe. C'est une instruction comprise uniquement par l'interpréteur IPython, sur lequel tourne Jupyter.

---

**Ce que fait `%time` précisément**

Il exécute la ligne normalement, puis affiche le temps que ça a pris :

```
CPU times: user 2.31 s, sys: 145 ms, total: 2.45 s
Wall time: 2.52 s
```

| Mesure | Signification |
|---|---|
| `user` | Temps passé par le CPU sur ton code |
| `sys` | Temps passé dans les appels système (lecture disque, réseau...) |
| `Wall time` | Le temps réel, celui de ta montre |

Ici c'est le `Wall time` qui t'intéresse. L'écart entre `total` et `Wall time` indique du temps d'attente — typiquement le téléchargement du modèle depuis le stockage MLflow.

**Point important** : la valeur de retour de la fonction est préservée. `%time` ne change rien au comportement, il ajoute juste une mesure.

---

**Pourquoi ici ?**

Parce que `test_model` fait des choses potentiellement lentes : charger un modèle depuis MLflow, appliquer le preprocessor, prédire sur tout le jeu de test. En production, savoir combien de temps prend une inférence est une vraie préoccupation MLOps.

---

**Un ou deux `%` ?**

```python
%time ma_fonction()      # mesure UNE ligne
```

```python
%%time                   # mesure TOUTE la cellule (doit être en 1ère ligne)
a = charger_donnees()
b = entrainer(a)
```

Cette logique vaut pour toutes les magics : `%` = ligne, `%%` = cellule.

---

**La variante `%timeit`**

```python
%timeit ma_fonction()
```

Celle-là exécute la fonction **plusieurs fois** et donne une moyenne avec écart-type. Plus fiable statistiquement, mais à éviter ici — tu ne veux pas relancer sept fois un test sur un gros dataset.

---

**Autres magics que tu croiseras dans le cours**

```python
!ls -la              # ← le "!" exécute une commande shell
%pwd                 # répertoire courant
%env                 # variables d'environnement
%load_ext autoreload # recharge les modules modifiés sans redémarrer le kernel
```

**Bonne intuition, mais il faut distinguer deux caractères différents !**

Tu as raison sur le principe pour `!`, mais `%time` n'est pas du bash.

---

### Les deux mécanismes

```python
!ls -la                    # ← BASH : exécuté par le shell du système
%time ma_fonction()        # ← IPython : exécuté par Jupyter lui-même
```

| Préfixe | Qui exécute ? | Exemple |
|---|---|---|
| `!` | Le **shell** (bash) de ta machine | `!pip install xgboost` |
| `%` | L'interpréteur **IPython** | `%time`, `%pwd` |

---

### Concrètement

`!ls` → Jupyter passe la commande à bash, récupère le texte affiché et te le montre. Bash n'a aucune idée de ce qu'est ton notebook.

`%time` → il n'existe **aucune commande bash appelée `time` au sens d'IPython**. C'est une fonctionnalité interne de Jupyter qui sait mesurer l'exécution d'une ligne **Python**. Bash ne pourrait pas mesurer une fonction Python, il ne connaît pas ton kernel.

C'est la différence clé : les magics `%` ont accès à **tout le contexte Python** de ton notebook (tes variables, tes fonctions), alors que `!` lance un processus externe complètement séparé.

---

### Le piège classique qui illustre bien la différence

```python
!cd 02-experiment-tracking      # ← ne marche PAS durablement
%cd 02-experiment-tracking      # ← marche
```

Avec `!`, bash ouvre un sous-processus, change de dossier dedans, puis le sous-processus meurt. Ton notebook n'a pas bougé.

Avec `%cd`, IPython change réellement le répertoire de travail du kernel. Ça persiste.

Même logique pour `!conda activate mlopszoomcamp` dans une cellule : ça n'aura aucun effet sur ton kernel. C'est pour ça qu'il faut sélectionner l'environnement dans VS Code plutôt que d'essayer de l'activer depuis une cellule.

---

### Et pour Claude Code

Oui, tu as raison, le `!` y joue bien le rôle d'échappement vers le shell. Le principe général est le même : **un caractère de préfixe pour sortir du langage courant et s'adresser à autre chose**.

Mais attention à ne pas transposer trop loin — le `%` de Jupyter, lui, n'a pas d'équivalent bash. Il reste dans le monde Python.

In [41]:
%time test_model(name=model_name, stage="Production", X_test=X_test, y_test=y_test)

CPU times: user 139 ms, sys: 44.6 ms, total: 183 ms
Wall time: 447 ms


{'rmse': 6.659623830022514}

In [42]:
%time test_model(name=model_name, stage="Staging", X_test=X_test, y_test=y_test)

CPU times: user 6.94 s, sys: 216 ms, total: 7.16 s
Wall time: 7.28 s


{'rmse': 6.881555517147188}

In [43]:
client.transition_model_version_stage(
    name=model_name,
    version=4,
    stage="Production",
    archive_existing_versions=True
)

<ModelVersion: creation_timestamp=1652971637398, current_stage='Production', description='The model version 4 was transitioned to Staging on 2022-05-19', last_updated_timestamp=1652972763255, name='nyc-taxi-regressor', run_id='b8904012c84343b5bf8ee72aa8f0f402', run_link=None, source='./mlruns/1/b8904012c84343b5bf8ee72aa8f0f402/artifacts/model', status='READY', status_message=None, tags={}, user_id=None, version=4>